# Top firms quantity & rev

1. Decide on a list of the top 15 firms by dispatch quantity
2. ⁠Calculate the quantity they’ve dispatched each
3. ⁠Calculate their revenue by multiplying it by the price at the time (I think this might mean I should definitely restrict to the 6-6:05pm) market to make it much much simpler

In [ ]:
# TABLE: DISPATCHLOAD

# Select only columns we need

selected_cols = ['SETTLEMENTDATE, DUID, AVAILABILITY, RAISEREG, LOWERREG'] # do you need RAISEREGENABLEMENTMAX/RAISEREGENABLEMENTMIN/LOWREGENABLEMENTMAX/LOWREGENABLEMENTMIN

# somthing before to load the data in, magic
dispatch_load_df = dispatch_load_df[[selected_cols]]

# group by duid and calculate the dispatch quantity

# join with the generator_info_df to get the Company name

# sort to get top 15 firms by dispatch quantity 

# create a column to calculate revenue
# which price are you going to use, bid or regional dispatch price RRP from DISPATCHPRICE table


# Bidding behaviours

4. Calculate the average bid over time, volume weighted and unweighted. To do this I need to do the original operation where I melted the bid volumes and prices together. I think volume weighted means price*volume/volume


In [ ]:
# TABLE: BIDPEROFFER_D
# Count the number of bids per day - Do you want to do this per bid type or total 
filtered_volume_bids['SETTLEMENT_DATE'] = pd.to_datetime(filtered_volume_bids['SETTLEMENT_DATE'])
filtered_volume_bids.groupby(filtered_volume_bids['SETTLEMENT_DATE'].dt.date).size()

# Filter the interval_time to 18:00:00-18:00:05
filtered_volume_bids['INTERVAL_DATETIME'] = pd.to_datetime(filtered_volume_bids['INTERVAL_DATETIME'])
filtered_volume_bids_by_time = filtered_volume_bids[(filtered_volume_bids['INTERVAL_DATETIME'].dt.time >= pd.to_datetime('18:00:00').time()) &
                                                 (filtered_volume_bids['INTERVAL_DATETIME'].dt.time <= pd.to_datetime('18:00:00').time())]
# Join the filtered_volumn_bids with the generator_info_df to get the Company name
joined_df = pd.merge(filtered_volume_bids_by_time, generator_info_df, on=[‘DUID’], how=’left’)

# Reshape the data such that each bid volume is on its own row, 
volume_bids = pd.melt(joined_df, id_vars=['INTERVAL_DATETIME'], 
                      value_vars=['BANDAVAIL1', 'BANDAVAIL2', 'BANDAVAIL3', 'BANDAVAIL4', 
                                  'BANDAVAIL5', 'BANDAVAIL6', 'BANDAVAIL7', 'BANDAVAIL8', 
                                  'BANDAVAIL9', 'BANDAVAIL10'],
                      var_name='BIDBAND', value_name='BIDVOLUME')


In [ ]:
# TABLE: BIDDAYOFFER_D
# Get the price bids - another similar operation for filtering Bid Type to LOWERREG and RAISEREG??

# Filter feather files for RaiseReg and LowerReg BIDTYPEs and save as parquet files
# Create the output directory if it doesn't already exist
output_dir = '/Volumes/T7/bid-price-filtered-2'
os.makedirs(output_dir, exist_ok=True)

file_list = glob.glob('/Volumes/T7/bid-price-data-sorted/*.feather')

# Sort alphabetically by filename
file_list.sort()

for file in file_list:
    print(f"Processing {file}...")
    # Read the Feather file
    df = pd.read_feather(file)
    
    # Filter to keep rows where BIDTYPE is either "RAISEREG" or "LOWERREG"
    filtered_price_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

    # Print the first few rows of the filtered DataFrame
    print("Filtered DataFrame head:")
    print(filtered_price_bids.head())

    # Get just the filename without the path
    base_filename = os.path.basename(file)  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000.feather"
    # Remove '.feather' extension
    filename_no_ext = os.path.splitext(base_filename)[0]  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000"

    # Construct the full output path in output_dir
    out_file = os.path.join(output_dir, filename_no_ext + ".parquet")

    # Save to Parquet
    filtered_price_bids.to_parquet(out_file, index=False)
    print(f"Finished processing {file} -> {out_file}\n")

In [ ]:
# TABLE: BIDDAYOFFER_D

# Don't need filtering by the interval as it applies to all dispatch intervals in the day
# Price bids are submitted daily and are applicable from the beggining of the market day. 


filtered_price_bids = pd.melt(filtered_price_bids, id_vars=['SETTLEMENTDATE'], 
                     value_vars=['PRICEBAND1', 'PRICEBAND2', 'PRICEBAND3', 'PRICEBAND4', 
                                 'PRICEBAND5', 'PRICEBAND6', 'PRICEBAND7', 'PRICEBAND8', 
                                 'PRICEBAND9', 'PRICEBAND10'],
                     var_name='BIDBAND', value_name='BIDPRICE')
filtered_price_bids.head()

# The first interval of the market day is 04:05:00, but we are analyising 18:00:00-18:05:00 so will add a
# column called APPLICABLEFROM to make it easier to match volume and price bids.
filtered_price_bids['APPLICABLEFROM'] = filtered_price_bids['SETTLEMENTDATE'] + timedelta(hours=18, minutes=5)
filtered_price_bids.head()


In [ ]:
# JOINING BIDS VOLUME and BIDS PRICE

# Create a column with the numerical value of bid band
volume_bids['BIDBAND'] = pd.to_numeric(volume_bids['BIDBAND'].str[9:])
volume_bids.head()

filtered_price_bids['BIDBAND'] = pd.to_numeric(price_bids['BIDBAND'].str[9:])
filtered_price_bids.head()

# Joining
bids = pd.merge_asof(volume_bids.sort_values('INTERVAL_DATETIME'), 
                     filtered_price_bids.sort_values('APPLICABLEFROM'), 
                     left_on='INTERVAL_DATETIME', right_on='APPLICABLEFROM',
                     by='BIDBAND')
bids.head()

# Plot the results.
fig = px.area(bids.sort_values('BIDBAND'), x='INTERVAL_DATETIME', 
              y='BIDVOLUME', color='BIDPRICE')
fig.update_layout(yaxis_title="MW")
fig.show()

References: https://github.com/UNSW-CEEM/NEMOSIS/blob/master/examples/generator_bidding_data.ipynb

# Demand estimation

5. ⁠Then I would like to focus on getting to the demand estimation stage as quickly as possible because it’s the ‘proper economics’ bit! This means I need to calculate the market share for each of the top n firms I choose over time

From Just Starting Out, We model demand at the monthly level because bids are tendered monthly. We don't do this right?

In [ ]:
import statsmodels.api as sm

X = bids, fully-loaded, part-loaded, positive ffr(would the last 3 make sense for your model, I am not sure)
X_sm = sm.add_constant(X_drop_first)
model = sm.OLS(y.astype(float),X_sm.astype(int))
results = model.fit()
results.summary()